In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")
%matplotlib inline
_ = plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    EmpfaengerID,
    drop_duplicate_columns,
    common_translate,
)

### Target Population Filtering

The patients in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
rec = data["recipient_et_id_et"].copy()
targetpop = pd.read_parquet(targetpop_data)
data = data[rec.isin(targetpop["recipient_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of recipients in the data ({rec.nunique()}) and target population ({targetpop["recipient_et_id_et"].nunique()})
            to {rec[rec.isin(targetpop["recipient_et_id_et"])].nunique()} in the processed data.
        """
    )
)
del targetpop, rec

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

Only {term}`ET` data is in this file, so no data processing was necessary at this step (see [](general:ic)).

## Domain Steps

For this file the plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

In this file there are the state changes for every organ waiting list (see [](general:rf)).

In [ ]:
display_long_data_doc(
    data,
    [
        "recipient_et_id_et",
    ],
    "date",
    "organ",
)

In [ ]:
sel = data["organ"] != "Ki"
rec_old = data["recipient_et_id_et"].nunique()
data = data[~sel].drop(columns="organ").copy()
rec_new = data["recipient_et_id_et"].nunique()
display(
    Markdown(
        f"""We limited the waiting list data to kidney data and removed the `organ` column.
                     This removed {sel.sum()} ({sel.sum()/sel.shape[0]:.2%}) rows and {rec_old-rec_new} ({(rec_old-rec_new)/rec_old:.2%}) recipients."""
    )
)
del sel, rec_old, rec_new

### Unit Conversions

The waiting codes in `waiting_state` were converted to the short representations and we applied the common translations (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

In [ ]:
newcol = data["waiting_state"].str.extract("([^-]{1,2}) - .+")
analysis = pd.concat([data["waiting_state"], newcol], axis=1)
analysis = (
    analysis.groupby(analysis.columns.to_list())
    .size()
    .sort_values()
    .rename("Replaced Count")
    .reset_index()
    .rename(columns={"waiting_state": "Replaced", 0: "Replaced with"})
    .set_index("Replaced")
)
data["waiting_state"] = newcol
display(analysis)
del newcol, analysis

### Consolidating Columns

No consolidation was necessary. (see [](general:crc))

## Intermediate Dataset

For this longitudinal dataset we recommend the `date` column as the time axis.

In [ ]:
indcols = ["recipient_et_id_et"]
data = data.sort_index(axis=1).sort_values(["date"], axis=0)
data = data.set_index(indcols)

In [ ]:
class EmpfaengerDringlichkeit(EmpfaengerID):
    date: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Measurement Date",
        description="When was the measurement taken?",
    )
    reason: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="State Change Reason",
        description="What was the reason for this state change?",
        isin=[
            "Other",
            "Medical (e.g. Infection)",
            "Transplanted",
            "On request of recipient (e.g. Holiday)",
            "On waiting list for living donor",
            "Shunt problems",
            "Temporarily NT, too bad for transplantation",
            "Incomplete recipient registration",
            "Psychological problems with risk of suicide",
            "Recipient unfit for transplantation",
            "(Uremic) Polyneuropathy",
            "Temporarily NT, too good for transplantation",
            "Recovered Recipient",
            "Kidney graft failure after simultaneous kidney/pancreas transplant",
            "Deceased while on the waiting list",
            "Wrong listing / administrative error",
        ],
    )
    waiting_state: Series[str] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Waiting State",
        description="What was the new state of the patient on this list?",
        isin=["NT", "T", "I", "HI", "HU", "FU", "R", "D"],
    )

    class Config:
        title = "Recipient Waiting List Dataset"
        description = "Each row represents a change on the waiting list for a recipient. The data is based on the 'element_empfaenger_dringlichkeit.csv' file. It contains data from the ET."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(EmpfaengerDringlichkeit, data)

In [ ]:
EmpfaengerDringlichkeit.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    EmpfaengerDringlichkeit.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)